# Технический экспорт фаз сердца в сцену OpenUSD

**Вопрос исследования.** Сохраняются ли геометрия, единицы длины и порядок наблюдённых фаз при переносе автоматических поверхностей сердца Ника в формат OpenUSD, пригодный для последующего просмотра в Omniverse? Экспорт и численная проверка повторным чтением выполнены. Сцену в Omniverse пока не просматривали, поэтому визуальное обнаружение анатомических дефектов не оценено.

OpenUSD здесь служит форматом трёхмерной сцены, а Omniverse — средой её просмотра. Экспортированная последовательность не является прогнозом по одной КТ или проверенным полем движения. [Учебный маршрут MONAI Physio](https://project-monai.github.io/monai-physio/tutorials.html#tutorial-5-vtk-surfaces-to-animated-usd) описывает перенос геометрии в USD.

## Источник поверхностей и смысл кадров

Источником стали существующие STL-поверхности `whole_heart`, полученные отдельной автоматической сегментацией TotalSegmentator v2 для **каждой из 12 фаз** 4D-КТ Ника. Фазы доступны как дискретные наблюдённые изображения; их номера задают порядок файлов, но не измеренное время ЭКГ. Экспорт не использует поле регистрации MP-03/04 и не переносит вершины одной опорной поверхности на остальные фазы.

Для исходной поверхности фазы 00 предыдущий контроль насчитал 307 036 треугольников и показал, что её координатные границы лежат в поле зрения КТ. Маска достигает верхней границы поля зрения; это связано с меньшим охватом 4D-КТ относительно статической КТ и само по себе не считается патологическим дефектом. Проверка поля зрения не подтверждает анатомическую точность маски. Исходные STL и полученная индивидуальная сцена остаются в локальном хранилище вне Git.

## Преобразование координат и контроль записи

Маршрут `WorkflowConvertVTKToUSD` преобразовал координаты из исходной системы RAS в миллиметрах в систему сцены с направленной вверх осью Y: `(x, y, z) мм → (x, z, −y) м`. Для сцены указаны `metersPerUnit = 1` и `upAxis = Y`. Значение физического масштаба проверяют вместе с фактическими числами координат: изменение только метаданных единиц после перевода точек в метры дало бы ошибку масштаба.

После записи OpenUSD повторно прочитал все 12 отсчётов. Для каждого сравнили число вершин и граней, центр и границы поверхности с ожидаемыми после преобразования. Максимальная ошибка центра составила **8,63 × 10⁻¹¹ м**, границ — **7,27 × 10⁻⁹ м**. Эти малые числа подтверждают точность сериализации координат в данном файле; они не измеряют правильность самой автоматической сегментации. Обезличенный [протокол численного контроля](mp09_nix_usd_visual_qc.json) содержит пофазные размеры и контрольные суммы STL.

## Дискретные фазы и ограничение анимации

В экспортированной сцене число вершин и топология меняются от фазы к фазе. Поэтому вершина с одинаковым номером в двух кадрах не обязана представлять одну анатомическую точку. Автоматическая интерполяция между такими кадрами не даёт подтверждённой промежуточной формы сердца. Номинальный порядок кадров также нельзя переводить в длительность кардиоцикла без принятой временной привязки.

Сцена технически пригодна для дискретного просмотра наблюдённых автоматических форм. Её можно использовать для поиска скачков, отражения, ошибки масштаба, ориентации или выхода за поле зрения, если каждый подозрительный кадр затем сверить с соответствующей КТ и маской. Сам просмотр без такой сверки не будет независимой анатомической валидацией и не допускает геометрию в FEM.

## Предыдущие тесты интерфейса и дальнейшее решение

На синтетической сфере интерфейс VTK→USD ранее записал три известных положения. Повторное чтение подтвердило один Mesh, отсчёты 0, 1 и 2 и по 74 вершины. Исходные смещения (0, 0, 0), (2, 0, 0) и (2, 1, 1,5) мм перешли в (0, 0, 0), (0,002, 0, 0) и (0,002, 0,0015, −0,001) м с ошибкой не более 5 × 10⁻⁶ м. Синтетический опыт проверял единицы и интерфейс, а не анатомию. Метка `timeCodesPerSecond = 2` в таком файле относится к настройке сцены; она не измеряет частоту сердечного цикла.

Ранний просмотр поверхности Georg охватывал лишь её STL и заголовок КТ, без экспорта реальной сцены Ника. В результате нынешний статус точен: **технический экспорт 12 фаз Ника выполнен, визуальный просмотр в Omniverse и независимая анатомическая проверка — нет**. Для научного использования необходимо сверить фазовые поверхности с исходными КТ, задокументировать найденные дефекты, а для непрерывного движения — отдельно проверить соответствие вершин и поле переноса. [Отчёт об экспорте](mp09_nix_usd_report.md) описывает условия использования; индивидуальный файл `runs/mp09_nix_observed_auto_phases.usdc` остаётся вне Git.

## Интерактивный просмотр наблюдённых фаз

Ячейка ниже показывает 12 отдельных автоматических поверхностей сердца Ника,
по которым создавалась сцена OpenUSD. Модель можно вращать и увеличивать;
ползунок выбирает фазу, кнопка последовательно показывает дискретные кадры.
Каждая поверхность упрощается только для вывода. Вершины разных фаз не имеют
подтверждённого соответствия, поэтому промежуточная анимация и скорость показа
не являются измеренным непрерывным движением. Это визуальный контроль
автоматических масок, а не проверка прогноза или результата регистрации.
Индивидуальная геометрия сохраняется только в локальном HTML вне Git.
Для запуска задайте `MONAI_PHYSIO_DERIVED_4D_ROOT` как локальный корень
производных 4D-КТ, содержащий `nix/products_v2/`.


In [ ]:
import os
from pathlib import Path
from IPython.display import HTML, display
from mp_visualize_motion import motion_figure

# Локальный корень производных 4D-КТ задаётся вне ноутбука.
root = Path(os.environ["MONAI_PHYSIO_DERIVED_4D_ROOT"])
source_dir = root / "nix" / "products_v2"
frames = [
    (f"{index:02d}", {"Отдельная автоматическая маска сердца":
                      source_dir / f"phase_{index:02d}" / "whole_heart.stl"})
    for index in range(12)
]
figure = motion_figure(
    frames,
    title="Ник: 12 отдельно сегментированных фаз сердца",
    subtitle="Наблюдённые КТ; дискретные автоматические маски без соответствия вершин.",
    colors={"Отдельная автоматическая маска сердца": "#c85a62"},
    opacity={"Отдельная автоматическая маска сердца": 0.96},
    same_topology=False,
    coordinate_system="RAS",
    reduction=0.985,
)
display(HTML(figure.to_html(full_html=False, include_plotlyjs=True)))
output = Path("runs/mp09_nix_observed_interactive_3d.html")
figure.write_html(str(output), include_plotlyjs=True)
print("Локальный интерактивный HTML:", output)